In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

events = spark.table("urban_mobility.silver.trip_events_clean")
trips = spark.table("urban_mobility.silver.trips_enriched")

requests_daily = (
    events.filter(F.col("event_type") == "TRIP_REQUESTED")
    .withColumn("date", F.to_date("event_timestamp"))
    .groupBy("date")
    .agg(F.count("*").alias("trip_requests"))
)

drivers_daily = (
    events.filter(F.col("event_type") == "DRIVER_ASSIGNED")
    .withColumn("date", F.to_date("event_timestamp"))
    .groupBy("date")
    .agg(F.countDistinct("driver_id").alias("active_drivers"))
)

trips_daily = (
    trips
    .withColumn("date", F.to_date("pickup_datetime"))
    .groupBy("date")
    .agg(
        F.sum(F.when(F.col("trip_status") == "COMPLETED", 1).otherwise(0)).alias("completed_trips"),
        F.sum(F.when(F.col("trip_status") == "CANCELLED", 1).otherwise(0)).alias("cancelled_trips"),
        F.round(F.sum(F.when(F.col("trip_status") == "COMPLETED", F.col("total_amount")).otherwise(0)), 2).alias("total_revenue"),
        F.round(F.avg(F.when(F.col("trip_status") == "COMPLETED", F.col("fare_amount"))), 2).alias("avg_fare"),
        F.round(F.avg(F.when(F.col("trip_status") == "COMPLETED", F.col("distance_km"))), 2).alias("avg_trip_distance_km"),
        F.round(F.avg(F.when(F.col("trip_status") == "COMPLETED", F.col("trip_duration_minutes"))), 2).alias("avg_duration_minutes"),
    )
)

hourly_requests = (
    events.filter(F.col("event_type") == "TRIP_REQUESTED")
    .withColumn("date", F.to_date("event_timestamp"))
    .withColumn("hour", F.hour("event_timestamp"))
    .groupBy("date", "hour")
    .agg(F.count("*").alias("hour_count"))
)
window_spec = Window.partitionBy("date").orderBy(F.desc("hour_count"))
peak_hour_daily = (
    hourly_requests
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .select("date", F.col("hour").alias("peak_demand_hour"))
)

daily_kpis = (
    requests_daily
    .join(trips_daily, "date", "left")
    .join(drivers_daily, "date", "left")
    .join(peak_hour_daily, "date", "left")
    .withColumn(
        "completion_rate",
        F.round(F.col("completed_trips") / (F.col("completed_trips") + F.col("cancelled_trips")), 4)
    )
    .orderBy("date")
)

daily_kpis.write.mode("overwrite").format("delta").saveAsTable("urban_mobility.gold.daily_mobility_kpis")

result = spark.table("urban_mobility.gold.daily_mobility_kpis")
print("gold.daily_mobility_kpis rows:", result.count())
result.orderBy(F.desc("trip_requests")).show(5)